In [1]:
#Import pandas, matplotlib.pyplot, and seaborn in the correct lines below
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
# the supplied CSV data file is the raw_data directory
df_2021 = pd.read_sas(r"../Raw data/LLCP2021.XPT", format="xport")

In [3]:
df_2021.head()

,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_FRTRES1,_VEGRES1,_FRUTSU1,_VEGESU1,_FRTLT1A,_VEGLT1A,_FRT16A,_VEG23A,_FRUITE1,_VEGETE1
0,1.0,1.0,b'01192021',b'01',b'19',b'2021',1100.0,b'2021000001',2.021000e+09,1.0,...,1.0,1.0,100.0,214.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79
1,1.0,1.0,b'01212021',b'01',b'21',b'2021',1100.0,b'2021000002',2.021000e+09,1.0,...,1.0,1.0,100.0,128.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79
2,1.0,1.0,b'01212021',b'01',b'21',b'2021',1100.0,b'2021000003',2.021000e+09,1.0,...,1.0,1.0,100.0,71.0,1.0,2.0,1.0,1.0,5.397605e-79,5.397605e-79
3,1.0,1.0,b'01172021',b'01',b'17',b'2021',1100.0,b'2021000004',2.021000e+09,1.0,...,1.0,1.0,114.0,165.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79
4,1.0,1.0,b'01152021',b'01',b'15',b'2021',1100.0,b'2021000005',2.021000e+09,1.0,...,1.0,1.0,100.0,258.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79


In [7]:
df_2021.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 438693 entries, 0 to 438692
Columns: 303 entries, _STATE to _VEGETE1
dtypes: float64(298), object(5)
memory usage: 1014.1+ MB


Below is a summary of all the variables I am keeping for training the model

Target variable:
1. _RFHYPE6 - Ever been told that blood pressure is high

Demograhics:
1. MARITAL - Marital status
2. _RACE - Race/ethnicity categories
3. _SEX - Respondent sex (constructed variable)
4. _AGEG5YR - Reported age in 5-year categories calculated variable
5. _AGE80 - Imputed age value collapsed above 80

Social determinants
1. _STATE - State Code
2. _METSTAT - Metropolitan Status
3. _URBSTAT - Urban/rural
4. EMPLOY1 - Employment status
5. _EDUCAG - Computed level of education completed categories
6. _INCOMG1 - Computed income categories
7. _HLTHPLN - Have any health insurance
8. PERSDOC3 - Have personal health care provider

Lifestyle and nutrition indicators 
1. _TOTINDA - Adults who reported doing physical activity or exercise in the past 30 days other than their regular job
2. _BMI5 - Computed BMI
3. _BMI5CAT - Computed BMI categories
4. _SMOKER3 - Computed smoking status
5. _CURECI1 - Current e-cig user calculated variable
6. DRNKANY5 - Computed adult having at least one drink of alcohol in the last 30 days
7. _FRUTSU1 - Total fruits consumed per day
8. _VEGESU1 - Total vegetables consumed per day
9. GRENDA1 - Computed dark green vegetables intake in times per day
10. FRNCHDA_ - Computed french fry intake in times per day

In [9]:
#Keeping target variable + independent variables 
columns_to_keep = ["_STATE","PERSDOC3",	"_RFHYPE6",	"MARITAL",	"EMPLOY1",	"_METSTAT",	"_URBSTAT",	"_RFHLTH",	"_PHYS14D",	"_HLTHPLN",	"_TOTINDA",	"_RACE",	"_SEX",	"_AGEG5YR",	"_AGE80",	"_BMI5",	"_BMI5CAT",	"_EDUCAG",	"_INCOMG1",	"_SMOKER3",	"_CURECI1",	"DRNKANY5",	"_FRUTSU1",	"_VEGESU1", "GRENDA1_", "FRNCHDA_"]

In [11]:
#Creating a new dataframe to only store the required variables 
df_2021_n = df_2021[columns_to_keep]

In [13]:
df_2021_n.head()

,_STATE,PERSDOC3,_RFHYPE6,MARITAL,EMPLOY1,_METSTAT,_URBSTAT,_RFHLTH,_PHYS14D,_HLTHPLN,...,_BMI5CAT,_EDUCAG,_INCOMG1,_SMOKER3,_CURECI1,DRNKANY5,_FRUTSU1,_VEGESU1,GRENDA1_,FRNCHDA_
0,1.0,1.0,1.0,1.0,7.0,1.0,1.0,2.0,3.0,1.0,...,1.0,2.0,3.0,3.0,1.0,2.0,100.0,214.0,5.700000e+01,4.300000e+01
1,1.0,2.0,2.0,9.0,8.0,1.0,1.0,1.0,1.0,1.0,...,NaN,4.0,9.0,4.0,1.0,2.0,100.0,128.0,1.400000e+01,5.397605e-79
2,1.0,2.0,2.0,3.0,7.0,1.0,1.0,1.0,1.0,1.0,...,3.0,2.0,2.0,4.0,1.0,2.0,100.0,71.0,5.397605e-79,1.400000e+01
3,1.0,1.0,2.0,1.0,7.0,1.0,1.0,1.0,1.0,1.0,...,4.0,2.0,5.0,4.0,1.0,1.0,114.0,165.0,1.000000e+01,5.700000e+01
4,1.0,1.0,1.0,1.0,8.0,2.0,1.0,2.0,3.0,1.0,...,3.0,1.0,2.0,4.0,1.0,2.0,100.0,258.0,1.000000e+02,2.900000e+01


In [15]:
df_2021_n.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 438693 entries, 0 to 438692
Data columns (total 26 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   _STATE    438693 non-null  float64
 1   PERSDOC3  438691 non-null  float64
 2   _RFHYPE6  438693 non-null  float64
 3   MARITAL   438688 non-null  float64
 4   EMPLOY1   435105 non-null  float64
 5   _METSTAT  431639 non-null  float64
 6   _URBSTAT  431639 non-null  float64
 7   _RFHLTH   438693 non-null  float64
 8   _PHYS14D  438693 non-null  float64
 9   _HLTHPLN  438693 non-null  float64
 10  _TOTINDA  438693 non-null  float64
 11  _RACE     438693 non-null  float64
 12  _SEX      438693 non-null  float64
 13  _AGEG5YR  438693 non-null  float64
 14  _AGE80    438693 non-null  float64
 15  _BMI5     391841 non-null  float64
 16  _BMI5CAT  391841 non-null  float64
 17  _EDUCAG   438693 non-null  float64
 18  _INCOMG1  438693 non-null  float64
 19  _SMOKER3  438693 non-null  float64
 20  _CUR

In [50]:
df_2021_n.describe().T

,count,mean,std,min,25%,50%,75%,max
_STATE,436781.0,30.738118,15.336655,1.000000e+00,20.0,31.0,41.0,78.0
PERSDOC3,436781.0,1.575700,0.881175,1.000000e+00,1.0,1.0,2.0,9.0
_RFHYPE6,436781.0,1.394095,0.488656,1.000000e+00,1.0,1.0,2.0,2.0
MARITAL,436778.0,2.393275,1.819099,1.000000e+00,1.0,1.0,3.0,9.0
EMPLOY1,433227.0,3.798452,2.871476,1.000000e+00,1.0,2.0,7.0,9.0
_METSTAT,429751.0,1.304118,0.460033,1.000000e+00,1.0,1.0,2.0,2.0
_URBSTAT,429751.0,1.144100,0.351191,1.000000e+00,1.0,1.0,1.0,2.0
_RFHLTH,436781.0,1.185338,0.537415,1.000000e+00,1.0,1.0,1.0,9.0
_PHYS14D,436781.0,1.608126,1.287197,1.000000e+00,1.0,1.0,2.0,9.0
_HLTHPLN,436781.0,1.362926,1.549699,1.000000e+00,1.0,1.0,1.0,9.0


In [17]:
df_2021_n.shape

(438693, 26)

In [19]:
#Target Variable 1: Have you ever been told that you have high blood pressure? 
df_2021_n['_RFHYPE6'].value_counts()

_RFHYPE6
1.0    264648
2.0    172133
9.0      1912
Name: count, dtype: int64

Note - 1 -No, 2 = Yes, and 9.0 = don't know/not sure/refused/missing. Drop all NA/Missing from the target variable 

In [21]:
#Keeping only non-missing observations for our target variable 
df_2021_n = df_2021_n[df_2021_n['_RFHYPE6']!= 9.0]

In [23]:
#Checking to make sure only missing observations were dropped
df_2021_n['_RFHYPE6'].value_counts()

_RFHYPE6
1.0    264648
2.0    172133
Name: count, dtype: int64

In [25]:
#Most of the variables have don't know/refused/missing special coded. These will need to replaced as NaN 

import numpy as np

cols = ['PERSDOC3','MARITAL','EMPLOY1','_RFHLTH','_PHYS14D','_HLTHPLN','_TOTINDA','_RFHYPE6','_RACE','_AGEG5YR','_EDUCAG','_INCOMG1','_SMOKER3','_CURECI1','DRNKANY5']
missing_codes = [9.0, 14.0]

df_2021_n[cols] = df_2021_n[cols].replace(missing_codes, np.nan)

In [27]:
df_2021_n.isna().sum()

_STATE          0
PERSDOC3      858
_RFHYPE6        0
MARITAL      4726
EMPLOY1      7998
_METSTAT     7030
_URBSTAT     7030
_RFHLTH      1075
_PHYS14D     9252
_HLTHPLN    16928
_TOTINDA      814
_RACE       10633
_SEX            0
_AGEG5YR    53371
_AGE80          0
_BMI5       46198
_BMI5CAT    46198
_EDUCAG      2290
_INCOMG1    93322
_SMOKER3    24639
_CURECI1    23666
DRNKANY5    26519
_FRUTSU1    50540
_VEGESU1    59506
GRENDA1_    43802
FRNCHDA_    44336
dtype: int64

In [29]:
#Count the number of missing values in each column and sort them.
missing = pd.concat([df_2021_n.isnull().sum(), 100 * df_2021_n.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by='count', ascending=True)

,count,%
_STATE,0,0.000000
_AGE80,0,0.000000
_SEX,0,0.000000
_RFHYPE6,0,0.000000
_TOTINDA,814,0.186363
PERSDOC3,858,0.196437
_RFHLTH,1075,0.246119
_EDUCAG,2290,0.524290
MARITAL,4726,1.082007
_URBSTAT,7030,1.609502


Note - There are several variables with NaN, with 21% of the income data missing. Will need to discuss with mentor on how to go about dealing with the missing observations. 

In [37]:
df_2021_n.info()

<class 'pandas.core.frame.DataFrame'>
Index: 436781 entries, 0 to 438692
Data columns (total 26 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   _STATE    436781 non-null  float64
 1   PERSDOC3  436781 non-null  float64
 2   _RFHYPE6  436781 non-null  float64
 3   MARITAL   436778 non-null  float64
 4   EMPLOY1   433227 non-null  float64
 5   _METSTAT  429751 non-null  float64
 6   _URBSTAT  429751 non-null  float64
 7   _RFHLTH   436781 non-null  float64
 8   _PHYS14D  436781 non-null  float64
 9   _HLTHPLN  436781 non-null  float64
 10  _TOTINDA  436781 non-null  float64
 11  _RACE     436781 non-null  float64
 12  _SEX      436781 non-null  float64
 13  _AGEG5YR  436781 non-null  float64
 14  _AGE80    436781 non-null  float64
 15  _BMI5     390583 non-null  float64
 16  _BMI5CAT  390583 non-null  float64
 17  _EDUCAG   436781 non-null  float64
 18  _INCOMG1  436781 non-null  float64
 19  _SMOKER3  436781 non-null  float64
 20  _CURECI1 

In [31]:
df_2021_n['_BMI5CAT'].value_counts()

_BMI5CAT
3.0    138325
4.0    130889
2.0    115111
1.0      6258
Name: count, dtype: int64

In [33]:
df_2021_n['_BMI5'].value_counts()

_BMI5
2663.0    4207
2746.0    3311
2441.0    3259
2744.0    3249
2712.0    2927
          ... 
5681.0       1
6328.0       1
7328.0       1
5964.0       1
5632.0       1
Name: count, Length: 3885, dtype: int64

Note: I currently have two variables, one continuous BMI variable and BMI Categories. Will need to decide which one is better suited for the model. 

BMI Categories
1 = Underweight, 2 = Normal Weight, 3 = Overweight, 4 = Obese 

In [35]:
df_2021_n[['_FRUTSU1','_VEGESU1','GRENDA1_', 'FRNCHDA_']].describe().T

,count,mean,std,min,25%,50%,75%,max
_FRUTSU1,386241.0,178.287455,691.237509,5.397605e-79,57.0,100.0,200.0,19800.0
_VEGESU1,377275.0,271.266400,1033.930858,5.397605e-79,114.0,167.0,229.0,39600.0
GRENDA1_,392979.0,78.829184,462.703124,5.397605e-79,14.0,43.0,71.0,9900.0
FRNCHDA_,392445.0,25.668109,172.499483,5.397605e-79,3.0,14.0,29.0,9900.0


Note - All the nutrition indicators are two implied decimal places 

In [37]:
#Creating a new target variable that is binary (0=No high NP, 1= High BP) 
df_2021_n['HIGHBP'] = df_2021_n['_RFHYPE6'].map({1: 0, 2: 1})

In [39]:
#Mean of BP Prevalance by age-group 
#Note - Group 14.0 is actually special code for missing/NA.
bp_age_means = df_2021_n.groupby('_AGEG5YR')[['HIGHBP']].mean()
bp_age_means

,HIGHBP
_AGEG5YR,
1.0,0.074813
2.0,0.107647
3.0,0.140707
4.0,0.181247
5.0,0.237019
6.0,0.296388
7.0,0.360210
8.0,0.424838
10.0,0.543119


In [41]:
#Mean of BP Prevalance by sex (1=Male, 2=Female)
bp_sex_means = df_2021_n.groupby('_SEX')[['HIGHBP']].mean()
bp_sex_means

,HIGHBP
_SEX,
1.0,0.414800
2.0,0.376151


In [43]:
#Mean of BP Prevalance by BMI Categories (1=Male, 2=Female)
bp_bmi_means = df_2021_n.groupby('_BMI5CAT')[['HIGHBP']].mean()
bp_bmi_means

,HIGHBP
_BMI5CAT,
1.0,0.249121
2.0,0.263632
3.0,0.393139
4.0,0.527195


In [47]:
#Mean of BP Prevalance by BMI Categories (1=Male, 2=Female)
bp_income_means = df_2021_n.groupby('_INCOMG1')[['HIGHBP']].mean()
bp_income_means

,HIGHBP
_INCOMG1,
1.0,0.491863
2.0,0.487642
3.0,0.436650
4.0,0.428246
5.0,0.378293
6.0,0.318133
7.0,0.264606


Note - As expected, we see increasing prevalance of high BP with older age, being male, being in the overweight/obese BMI Category, and also in the lower income levels. 

In [56]:
#Saving the current dataframe as a csv file
df_2021_n.to_csv('../data/brfss_2021_clean.csv', index=False)